# QC: session-to-session connectome similarity, split four ways

**Exploratory QC, not the primary analysis.** For each network and each
measure, this correlates every pair of session connectome edge vectors
(Fisher-z transformed first) and splits the pairs into four bins by
same/different subject x same/different dataset. It has no usable-data gate
(the FD / duration criteria are still open — CLAUDE.md, "Still open"), so
treat it as a sanity check, not a headline result. Deliberately kept out of
the montage figure and out of `run-group-stats` (CLAUDE.md, "Respect the
analysis hierarchy").

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

output_dir = Path(os.environ.get("OUTPUT_DATA_DIR", "../output_data")).resolve()
figures_base = Path(os.environ.get("FIGURES_DIR", output_dir / "figures")).resolve()

# invoke.yaml is not exposed as an env var by run-notebooks, so read the
# project root config directly for the parcellation and network order. The
# same root also holds analysis/, which is not on sys.path when nbconvert
# launches the kernel from an arbitrary cwd, so add it before importing.
project_root = output_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from analysis.similarity import (  # noqa: E402
    collect_pair_values,
    common_edge_mask,
    discover_connectome_files,
    fisher_z,
    load_stacked_measure,
    pair_bins,
    similarity_matrix,
    summarize_bins,
)

# Not a montage panel, so there is no panel_size box to render into — this
# notebook picks its own free figure size instead.
figure_dir = figures_base / "qc_similarity"
figure_dir.mkdir(parents=True, exist_ok=True)

with open(project_root / "invoke.yaml") as handle:
    invoke_config = yaml.safe_load(handle)

PARCELLATION = invoke_config["parcellation"]
NETWORK_ORDER = invoke_config["parcellations"][PARCELLATION]["network_order"]
MEASURES = invoke_config.get("connectome_measures", ["pearson", "partial_ledoitwolf"])

connectome_dir = output_dir / "connectomes"
paths, skipped = discover_connectome_files(connectome_dir, PARCELLATION)
for path, reason in skipped:
    print(f"⚠️  skipping {path.name}: {reason}")
print(f"📂 {len(paths)} connectome file(s) for parcellation={PARCELLATION}: "
      f"{[p.stem for p in paths]}")

In [ ]:
# Colors keyed by bin label, shared across every panel and both measures so
# the four distributions stay visually comparable figure to figure.
BIN_COLORS = {
    "within-subject / within-dataset": "#1b9e77",
    "within-subject / between-dataset": "#7570b3",
    "between-subject / within-dataset": "#d95f02",
    "between-subject / between-dataset": "#999999",
}

summary_rows = []

for measure in MEASURES:
    fig, axes = plt.subplots(3, 3, figsize=(12, 10), layout="constrained")
    fig.suptitle(f"Session-pair connectome similarity — {measure}")

    for network, ax in zip(NETWORK_ORDER, axes.flat):
        index_frame, matrix = load_stacked_measure(paths, measure, network)
        z_matrix = fisher_z(matrix)
        valid = common_edge_mask(z_matrix)
        similarity = similarity_matrix(z_matrix)
        bins = pair_bins(index_frame)
        values_by_bin = collect_pair_values(similarity, bins)

        summary = summarize_bins(values_by_bin)
        summary["measure"] = measure
        summary["network"] = network
        summary["n_edges_valid"] = int(valid.sum())
        summary["n_edges_total"] = int(valid.size)
        summary_rows.append(summary)

        all_values = np.concatenate([v for v in values_by_bin.values() if len(v)])
        if len(all_values):
            bin_edges = np.linspace(all_values.min(), all_values.max(), 40)
        else:
            bin_edges = np.linspace(-1, 1, 40)
        for bin_label, values in values_by_bin.items():
            if len(values) == 0:
                continue
            counts, edges = np.histogram(values, bins=bin_edges, density=True)
            ax.stairs(counts, edges, color=BIN_COLORS[bin_label], label=bin_label)

        ax.set_title(network, fontsize=10)
        ax.set_yticks([])
        ax.annotate(f"edges {valid.sum()}/{valid.size}", xy=(0.02, 0.95),
                    xycoords="axes fraction", fontsize=7, va="top", color="0.4")

    axes.flat[0].legend(fontsize=7, loc="upper right")
    out_path = figure_dir / f"{measure}_similarity.png"
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f"✅ wrote {out_path}")

In [ ]:
summary_frame = pd.concat(summary_rows, ignore_index=True)
summary_frame = summary_frame[
    ["measure", "network", "bin", "n", "median", "q25", "q75", "mean", "sd",
     "n_edges_valid", "n_edges_total"]
]
summary_path = figure_dir / "pair_summary.tsv"
summary_frame.to_csv(summary_path, sep="\t", index=False)
print(f"✅ wrote {summary_path}")
summary_frame.head(12)